
## Octagon Oracle — Model Training Script
Run this script first to train and save the model.
Usage: python train_model.py
Dataset: preprocessed_data.csv from https://www.kaggle.com/datasets/rajeevw/ufcdata
Place the file at: dataset/preprocessed_data.csv

In [16]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, roc_auc_score)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────
# 1. LOAD DATA
# ──────────────────────────────────────────────

In [17]:
DATA_PATH = os.path.join('dataset', 'preprocessed_data.csv')

def load_data(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\n[ERROR] Dataset not found at '{path}'.\n"
            "Download 'preprocessed_data.csv' from:\n"
            "  https://www.kaggle.com/datasets/rajeevw/ufcdata\n"
            "Place it in the 'dataset/' folder and re-run."
        )
    df = pd.read_csv(path)
    print(f"[OK] Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")
    return df

# ──────────────────────────────────────────────
# 2. PREPROCESSING
# ──────────────────────────────────────────────

In [18]:
def preprocess(df):
    # Keep only Red/Blue outcomes (drop rare No Contest / Draw)
    df = df[df['Winner'].isin(['Red', 'Blue'])].copy()
    print(f"[OK] Filtered to Red/Blue only: {df.shape[0]} rows")

    # Target: 1 = Red wins, 0 = Blue wins
    df['target'] = (df['Winner'] == 'Red').astype(int)

    # ── Differential features (matchup-aware) ──
    num_pairs = [
        ('R_avg_SIG_STR_pct',  'B_avg_SIG_STR_pct',  'str_acc_diff'),
        ('R_avg_TD_pct',        'B_avg_TD_pct',        'td_acc_diff'),
        ('R_avg_SUB_ATT',       'B_avg_SUB_ATT',       'sub_att_diff'),
        ('R_avg_CTRL',          'B_avg_CTRL',          'ctrl_diff'),
        ('R_avg_SIG_STR_att',   'B_avg_SIG_STR_att',   'sig_str_att_diff'),
        ('R_avg_SIG_STR_landed','B_avg_SIG_STR_landed','sig_str_landed_diff'),
        ('R_avg_TD_att',        'B_avg_TD_att',        'td_att_diff'),
        ('R_avg_TD_landed',     'B_avg_TD_landed',     'td_landed_diff'),
        ('R_wins',              'B_wins',              'win_diff'),
        ('R_losses',            'B_losses',            'loss_diff'),
        ('R_win_by_KO',         'B_win_by_KO',         'ko_diff'),
        ('R_win_by_sub',        'B_win_by_sub',        'sub_diff'),
        ('R_win_by_Decision_Unanimous','B_win_by_Decision_Unanimous','dec_diff'),
        ('R_Height_cms',        'B_Height_cms',        'height_diff'),
        ('R_Reach_cms',         'B_Reach_cms',         'reach_diff'),
        ('R_age',               'B_age',               'age_diff'),
        ('R_avg_opp_SIG_STR_pct','B_avg_opp_SIG_STR_pct','opp_str_diff'),
    ]

    for r_col, b_col, new_col in num_pairs:
        if r_col in df.columns and b_col in df.columns:
            df[new_col] = df[r_col].fillna(0) - df[b_col].fillna(0)

    # Win ratio features
    for corner in ['R', 'B']:
        wins_col   = f'{corner}_wins'
        losses_col = f'{corner}_losses'
        ratio_col  = f'{corner}_win_ratio'
        if wins_col in df.columns and losses_col in df.columns:
            total = df[wins_col].fillna(0) + df[losses_col].fillna(0)
            df[ratio_col] = np.where(total > 0, df[wins_col].fillna(0) / total, 0.5)

    if 'R_win_ratio' in df.columns and 'B_win_ratio' in df.columns:
        df['win_ratio_diff'] = df['R_win_ratio'] - df['B_win_ratio']

    # Experience level (total fights)
    for corner in ['R', 'B']:
        wins_col   = f'{corner}_wins'
        losses_col = f'{corner}_losses'
        exp_col    = f'{corner}_experience'
        if wins_col in df.columns and losses_col in df.columns:
            df[exp_col] = df[wins_col].fillna(0) + df[losses_col].fillna(0)

    if 'R_experience' in df.columns and 'B_experience' in df.columns:
        df['experience_diff'] = df['R_experience'] - df['B_experience']

    # Stance matchup
    if 'R_Stance' in df.columns and 'B_Stance' in df.columns:
        df['stance_match'] = (df['R_Stance'] == df['B_Stance']).astype(int)
        df['R_is_southpaw'] = (df['R_Stance'] == 'Southpaw').astype(int)
        df['B_is_southpaw'] = (df['B_Stance'] == 'Southpaw').astype(int)

    # Weight class encoding
    if 'weight_class' in df.columns:
        wc_dummies = pd.get_dummies(df['weight_class'], prefix='wc')
        df = pd.concat([df, wc_dummies], axis=1)

    # Title bout flag
    if 'title_bout' in df.columns:
        df['title_bout'] = df['title_bout'].astype(int)

    return df

# ──────────────────────────────────────────────
# 3. FEATURE SELECTION
# ──────────────────────────────────────────────

In [19]:
FEATURE_COLS = [
    'str_acc_diff', 'td_acc_diff', 'sub_att_diff', 'ctrl_diff',
    'sig_str_att_diff', 'sig_str_landed_diff', 'td_att_diff', 'td_landed_diff',
    'win_diff', 'loss_diff', 'ko_diff', 'sub_diff', 'dec_diff',
    'height_diff', 'reach_diff', 'age_diff', 'opp_str_diff',
    'win_ratio_diff', 'experience_diff',
    'stance_match', 'R_is_southpaw', 'B_is_southpaw',
]

def get_features(df):
    available = [c for c in FEATURE_COLS if c in df.columns]
    wc_cols = [c for c in df.columns if c.startswith('wc_')]
    available += wc_cols
    if 'title_bout' in df.columns:
        available.append('title_bout')
    print(f"[OK] Using {len(available)} features")
    return available

# ──────────────────────────────────────────────
# 4. TRAIN AND EVALUATE
# ──────────────────────────────────────────────

In [20]:

def train_and_evaluate(df, feature_cols):
    X = df[feature_cols].fillna(0)
    y = df['target']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10,
                                                       min_samples_split=5, random_state=42),
        'XGBoost':             XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                                              subsample=0.8, colsample_bytree=0.8,
                                              eval_metric='logloss',
                                              random_state=42),
    }

    results = {}
    best_name, best_score, best_model = None, 0, None

    print("\n" + "="*55)
    print("  MODEL EVALUATION RESULTS")
    print("="*55)

    for name, model in models.items():
        X_tr = X_train_sc if name == 'Logistic Regression' else X_train
        X_te = X_test_sc  if name == 'Logistic Regression' else X_test

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(model, X_tr, y_train, cv=cv, scoring='accuracy')

        model.fit(X_tr, y_train)
        y_pred  = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]

        acc    = accuracy_score(y_test, y_pred)
        auc    = roc_auc_score(y_test, y_proba)
        report = classification_report(y_test, y_pred,
                                       target_names=['Blue wins', 'Red wins'])

        results[name] = {'accuracy': acc, 'auc': auc, 'cv_mean': cv_scores.mean(),
                         'cv_std': cv_scores.std(), 'model': model,
                         'y_pred': y_pred, 'y_proba': y_proba}

        print(f"\n{name}")
        print(f"  Test Accuracy : {acc*100:.2f}%")
        print(f"  AUC Score     : {auc:.4f}")
        print(f"  CV Accuracy   : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
        print(f"\n{report}")

        if auc > best_score:
            best_score = auc
            best_name  = name
            best_model = model

    print(f"\n[BEST MODEL] {best_name} (AUC: {best_score:.4f})")
    return best_name, best_model, scaler, X_test, y_test, results, feature_cols

# ──────────────────────────────────────────────
# 5. SAVE ARTIFACTS
# ──────────────────────────────────────────────

In [21]:
def save_artifacts(model, scaler, feature_cols, best_name):
    os.makedirs('model', exist_ok=True)
    joblib.dump(model,        'model/ufc_model.pkl')
    joblib.dump(scaler,       'model/scaler.pkl')
    joblib.dump(feature_cols, 'model/feature_cols.pkl')

    with open('model/model_info.txt', 'w') as f:
        f.write(f"Best Model: {best_name}\n")
        f.write(f"Features used: {len(feature_cols)}\n")
        f.write(f"Feature list:\n")
        for fc in feature_cols:
            f.write(f"  - {fc}\n")

    print("\n[OK] Saved: model/ufc_model.pkl")
    print("[OK] Saved: model/scaler.pkl")
    print("[OK] Saved: model/feature_cols.pkl")

# ──────────────────────────────────────────────
# 6. MAIN
# ──────────────────────────────────────────────

In [22]:
if __name__ == '__main__':
    print("\n" + "="*55)
    print("  OCTAGON ORACLE — MODEL TRAINING")
    print("="*55 + "\n")

    df_raw    = load_data(DATA_PATH)
    df        = preprocess(df_raw)
    feat_cols = get_features(df)
    best_name, best_model, scaler, X_test, y_test, results, feat_cols = \
        train_and_evaluate(df, feat_cols)
    save_artifacts(best_model, scaler, feat_cols, best_name)

    print("\n[DONE] Training complete. Run: python app.py")


  OCTAGON ORACLE — MODEL TRAINING

[OK] Loaded dataset: 5902 rows × 160 columns
[OK] Filtered to Red/Blue only: 5902 rows
[OK] Using 17 features

  MODEL EVALUATION RESULTS

Logistic Regression
  Test Accuracy : 67.15%
  AUC Score     : 0.6451
  CV Accuracy   : 67.74% ± 0.90%

              precision    recall  f1-score   support

   Blue wins       0.48      0.10      0.16       385
    Red wins       0.68      0.95      0.80       796

    accuracy                           0.67      1181
   macro avg       0.58      0.52      0.48      1181
weighted avg       0.62      0.67      0.59      1181


Random Forest
  Test Accuracy : 68.33%
  AUC Score     : 0.6485
  CV Accuracy   : 68.46% ± 0.83%

              precision    recall  f1-score   support

   Blue wins       0.58      0.10      0.17       385
    Red wins       0.69      0.96      0.80       796

    accuracy                           0.68      1181
   macro avg       0.64      0.53      0.49      1181
weighted avg       0.65